# Instance-based vs model-based learning: interactive

Run both cells with **Shift + Enter**. The second cell opens a small app inside the notebook.

**Try:**
- Move the new student around. Where do the two approaches disagree?
- Change *k*. With *k* = 1, a single unusual student can flip the answer; with a larger *k*, the vote is steadier.
- Notice that the learned boundary never moves: model-based learning learned it once. The neighbours change every time: instance-based learning works only when asked.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Example data: IQ and CGPA of 60 students, and whether each was placed (1) or not (0)
rng = np.random.default_rng(4)
iq, cgpa = rng.uniform(75, 135, 60), rng.uniform(5.0, 9.8, 60)
placed = (0.04 * iq + cgpa + rng.normal(0, 0.35, 60) > 11.8).astype(int)
X = np.column_stack([iq, cgpa])

scaler = StandardScaler().fit(X)           # IQ and CGPA on the same scale, so distances are fair
Xs = scaler.transform(X)
logreg = LogisticRegression().fit(Xs, placed)   # model-based: learns a boundary once

def update(new_iq, new_cgpa, k):
    """Classify one new student both ways and draw the picture."""
    q = scaler.transform([[new_iq, new_cgpa]])
    knn = KNeighborsClassifier(n_neighbors=k).fit(Xs, placed)   # instance-based: just stores the data
    _, idx = knn.kneighbors(q)
    votes = int(placed[idx[0]].sum())
    knn_says = "Placed" if knn.predict(q)[0] else "Not placed"
    log_says = "Placed" if logreg.predict(q)[0] else "Not placed"

    fig = go.Figure()
    for label, colour, name in [(1, "#54A24B", "placed"), (0, "#E45756", "not placed")]:
        m = placed == label
        fig.add_trace(go.Scatter(x=iq[m], y=cgpa[m], mode="markers", name=name, marker=dict(color=colour, size=9)))
    for i in idx[0]:
        fig.add_trace(go.Scatter(x=[new_iq, iq[i]], y=[new_cgpa, cgpa[i]], mode="lines", showlegend=False,
                                 line=dict(color="#4C78A8", dash="dot")))
    w, b = logreg.coef_[0], logreg.intercept_[0]
    line_iq = np.linspace(75, 135, 50)
    line_cg = -(w[0] * (line_iq - scaler.mean_[0]) / scaler.scale_[0] + b) / w[1] * scaler.scale_[1] + scaler.mean_[1]
    fig.add_trace(go.Scatter(x=line_iq, y=line_cg, mode="lines", name="learned boundary", line=dict(color="#B279A2", width=3)))
    fig.add_trace(go.Scatter(x=[new_iq], y=[new_cgpa], mode="markers", name="new student",
                             marker=dict(symbol="star", size=18, color="#4C78A8", line=dict(color="black", width=1))))
    fig.update_layout(xaxis=dict(title="IQ", range=[73, 137]), yaxis=dict(title="CGPA", range=[4.8, 10]),
                      template="simple_white", height=480, margin=dict(t=20))
    summary = (f"Instance-based (KNN, k = {k}): {votes} of {k} nearest were placed, so {knn_says}.   "
               f"Model-based (boundary): {log_says}.")
    return fig, summary

In [ ]:
from dash import Dash, dcc, html, Input, Output

app = Dash(__name__)
app.layout = html.Div([
    html.Div("New student's IQ"), dcc.Slider(75, 135, 1, value=94, id="iq", marks=None, tooltip={"always_visible": True}),
    html.Div("New student's CGPA"), dcc.Slider(5, 10, 0.1, value=8.3, id="cgpa", marks=None, tooltip={"always_visible": True}),
    html.Div("k (how many neighbours vote)"), dcc.Slider(1, 15, 2, value=3, id="k"),
    html.B(id="summary"), dcc.Graph(id="plot"),
])

@app.callback(Output("plot", "figure"), Output("summary", "children"),
              Input("iq", "value"), Input("cgpa", "value"), Input("k", "value"))
def on_change(new_iq, new_cgpa, k):
    return update(new_iq, new_cgpa, k)

app.run(jupyter_mode="inline", port=8056)